In [ ]:

import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt 
import subprocess
import matplotlib as mpl
import os
import re
import shlex
# import rcr_Suport as rcr
import copy
import shutil
import time
import shutil
from config import load_case, to_wsl

In [ ]:

def clean_previous_runs(output_dir):
    """
    Deletes all folders inside output_dir that match the pattern 'run_iter<number>'.
    """
    if not os.path.exists(output_dir):
        print(f"Directory does not exist: {output_dir}")
        return

    pattern = re.compile(r"run_iter\d+")

    removed = 0
    for item in os.listdir(output_dir):
        full_path = os.path.join(output_dir, item)

        if os.path.isdir(full_path) and pattern.fullmatch(item):
            try:
                shutil.rmtree(full_path)
                print(f"Removed folder: {full_path}")
                removed += 1
            except Exception as e:
                print(f"Failed to remove {full_path}: {e}")

    if removed == 0:
        print("No previous run folders found.")
    else:
        print(f"Cleanup complete. Removed {removed} folders.")


def write_in_file(path, RCR1, RCR2, RCR3, RCR4): 
    
    with open(path, 'r') as file1:
        content = file1.read()
    
    rcr_data = {
        'RCR_0': RCR1,
        'RCR_1': RCR2,
        'RCR_2': RCR3,
        'RCR_3': RCR4
    }
    
    import re

    for name, values in rcr_data.items():
        pattern = rf'(DATATABLE {name} LIST\n).*?(ENDDATATABLE)'
        replacement = (
            f'DATATABLE {name} LIST\n'
            f'0.0 {values[0]:.3f}\n'
            f'0.0 {values[1]:.3e}\n'
            f'0.0 {values[2]:.3f}\n'
            f'ENDDATATABLE'
        )
        content, count = re.subn(pattern, replacement, content, flags=re.DOTALL)
        if count != 1:
            raise ValueError(f"expected one {name} block in {path}, found {count}")
    
    with open(path , 'w') as file2:
        file2.write(content)
        
def time_steps_per_cycle(dt_time, max_time):
    if dt_time <= 0 or max_time <= 0:
        raise ValueError("dt_time and max_time must be positive")
    steps = int(round(max_time / dt_time))
    if steps < 1:
        raise ValueError("one cardiac cycle must contain at least one step")
    return steps

def write_in_file_options(path, dt_time, max_time, number_cycles): 
    
    with open(path, 'r') as file1:
        content = file1.read()

    pattern = r'^SOLVEROPTIONS.*$'
    number_steps = time_steps_per_cycle(dt_time, max_time) * number_cycles
    replacement = f"SOLVEROPTIONS {dt_time:.12g} 1 {number_steps} 2 INFLOW FLOW 1.0e-5 1 1"
    content, count = re.subn(pattern, replacement, content, flags=re.MULTILINE)
    if count != 1:
        raise ValueError(f"expected one SOLVEROPTIONS line in {path}, found {count}")

    with open(path , 'w') as file2:
        file2.write(content)

def delete_old(path, namerS):
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path) and item.startswith(namerS):
            print(item_path)
            shutil.rmtree(item_path)

def run_simulation_ubuntu(path, output_dir=None):
    if output_dir is None:
        raise ValueError("output_dir is required")

    solver = cfg["paths"]["onedsolver_bin"]
    command = (
        f"cd {shlex.quote(output_dir)} && "
        f"mpirun -np {cfg['n_procs']} {shlex.quote(solver)} {shlex.quote(path)}"
    )
    subprocess.run(["wsl", "bash", "-lc", command], check=True)
    return 0

def outlet_pressure_read(out_name, output_dir_windows, dt_time, max_time, number_cycles, max_retries=2, wait_seconds=30):

    start_dt = time_steps_per_cycle(dt_time, max_time) * (number_cycles - 1)

    pressure_path = os.path.join(output_dir_windows, out_name + "_pressure.dat")
    flow_path     = os.path.join(output_dir_windows, out_name + "_flow.dat")

    for attempt in range(1, max_retries + 1):
        try:
            data_p = np.loadtxt(pressure_path)
            pressure = data_p[-1, start_dt:]

            data_f = np.loadtxt(flow_path)
            flow = data_f[-1, start_dt:]

            return pressure, flow
        
        except Exception as e:
            print(f"  Failed to read {out_name}.")

            if attempt < max_retries:
                time.sleep(wait_seconds)
            else:
                print("  Maximum retries reached. Raising exception.")
                raise

def mesure_pressure_characteristics(pressure_iterac):
    Pmax_it = np.max(pressure_iterac)
    Pmin_it = np.min(pressure_iterac)
    Pmean_it = 2*Pmin_it/3 + (1/3) * Pmax_it
    Pmax_idx_it = np.argmax(pressure_iterac)
    Pmin_idx_it = np.argmin(pressure_iterac)
    return Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it
        
def compute_dL_ds(Pmax_index, Pmin_index, Pmax, Psys, Pmin, Pdia, Pmean, Pavg, n, dt):
    dL_ds = np.zeros((3, n))
    dL_ds[0, Pmax_index] = 1.0 / Psys
    dL_ds[1, Pmin_index] = 1.0 / Pdia
    triangle_profile = np.ones(n) 
    triangle_profile[0] = 0.5
    triangle_profile[-1] = 0.5
    dL_ds[2, :] = (dt) * triangle_profile / Pmean
    return dL_ds

def compute_dN_ds(n, Rp, C, dt):
    dN_ds = np.zeros((n, n))
    main = 0.5 + C*Rp/dt
    sub  = 0.5 - C*Rp/dt
    dN_ds[np.arange(n), np.arange(n)] = main
    dN_ds[np.arange(1,n), np.arange(0,n-1)] = sub
    return dN_ds

def compute_dN_dphi(n, p, flow, Rp, Rs, C, dt): 
    dN_dphi = np.zeros((n, 3))
    for k in range(n):
        if k == 0:
            dq    = 0.0
            dp    = 0.0
            q_avg = flow[0]
        else:
            j     = k - 1
            dq    = (flow[k] - flow[j]) / dt
            dp    = (p[k]    - p[j])    / dt
            q_avg = 0.5 * (flow[k] + flow[j])

        dN_dphi[k, 0] = - q_avg - C * Rp * dq
        dN_dphi[k, 1] = C * dp - q_avg - C * Rs * dq
        dN_dphi[k, 2] = Rp * dp - Rs * Rp * dq

    return dN_dphi

def L2_mesure(Pmax_it, Pmin_it, Pmean_it, Psys, Pdia, Pmean):
    L = np.array([
        (Pmax_it - Psys)  / Psys,
        (Pmin_it - Pdia)  / Pdia,
        (Pmean_it - Pmean) / Pmean
    ])
    return L, np.linalg.norm(L)

# Rs must stay in [Rs_ratio_min, Rs_ratio_max] * Rp
RS_RATIO_MIN = 0.01   # Rs >= 1% of Rp
RS_RATIO_MAX = 0.80   # Rs <= 80% of Rp  (relaxed – was 0.60, was binding for out1)

# Proportional-rescale is used when L2 > this threshold.
# Newton-Raphson is unreliable far from the solution and can push in the wrong
# direction; proportional scaling is always well-directed.
NEWTON_L2_THRESHOLD = 0.08   # was 0.25 — lowered so Newton only runs near convergence

def _apply_rcr_constraints(Rs_new, Rp_new, C_new):
    """Enforce physical bounds on RCR parameters."""
    Rp_new = np.clip(Rp_new, 100.0,  200000.0)
    C_new  = np.clip(C_new,  1e-7,   1e-1)
    Rs_new = np.clip(Rs_new,
                     RS_RATIO_MIN * Rp_new,
                     RS_RATIO_MAX * Rp_new)
    return Rs_new, Rp_new, C_new

def update_rcr_newton_raphson(Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it,
                               p, q, dt, Rs, Rp, C,
                               Psys_obj, Pdia_obj, Pmean_obj, learning_rate=0.1):

    L, L2_norm = L2_mesure(Pmax_it, Pmin_it, Pmean_it, Psys_obj, Pdia_obj, Pmean_obj)

    # ── Proportional-rescale step ────────────────────────────────────────────
    # Used whenever L2 > NEWTON_L2_THRESHOLD.  Newton is unreliable far from
    # the solution (can point in the wrong direction); direct scaling is always
    # well-directed because it drives each pressure component toward its target.
    if L2_norm > NEWTON_L2_THRESHOLD:
        # Scale total resistance by mean-pressure ratio (primary driver of Pmean)
        scale_mean = np.clip(Pmean_obj / Pmean_it, 0.5, 2.5)
        # Additionally correct systolic bias: if Psys is the dominant error,
        # nudge Rs (which shapes the systolic peak) separately.
        err_sys  = (Pmax_it - Psys_obj) / Psys_obj   # negative → need more Rs
        scale_rs = np.clip(1.0 - err_sys, 0.5, 2.5)  # >1 when Psys too low

        Rp_new = Rp * scale_mean
        Rs_new = Rs * scale_rs
        C_new  = C  / scale_mean   # keep Rp*C windkessel time constant
        print(f"    [rescale] scale_mean={scale_mean:.3f}, scale_rs={scale_rs:.3f}")
        return _apply_rcr_constraints(Rs_new, Rp_new, C_new)

    # ── Newton-Raphson step (only used when L2 <= NEWTON_L2_THRESHOLD) ───────
    n = len(p)
    dL_ds    = compute_dL_ds(Pmax_idx_it, Pmin_idx_it,
                              Pmax_it, Psys_obj, Pmin_it, Pdia_obj, Pmean_it, Pmean_obj, n, dt)
    dN_ds    = compute_dN_ds(n, Rp, C, dt)
    dN_dphi  = compute_dN_dphi(n, p, q, Rp, Rs, C, dt)

    dL_dphi = np.zeros((3, 3))

    X = np.linalg.solve(dN_ds, dN_dphi)
    J = -dL_ds @ X + dL_dphi

    phi = np.array([[Rs], [Rp], [C]])

    JTJ = J.T @ J
    lambda_reg = 1e-2
    damping = lambda_reg * np.diag(np.maximum(np.diag(JTJ), 1e-10))
    delta = np.linalg.solve(JTJ + damping, J.T @ L.reshape(-1, 1))

    # Cap each component to ±30% of its current value
    for i, val in enumerate([Rs, Rp, C]):
        max_step = 0.3 * abs(val)
        delta[i, 0] = np.clip(delta[i, 0], -max_step, max_step)

    phi_new = phi - learning_rate * delta
    Rs_new, Rp_new, C_new = phi_new.flatten()

    return _apply_rcr_constraints(Rs_new, Rp_new, C_new)


In [ ]:
# Paper algorithms use the public config interface. All clinical values and paths
# come from git-ignored config/local.py; copy config/local_example.py for its schema.
from config import load_case

CASE = os.environ.get("AORTA_CASE")
if not CASE:
    raise RuntimeError("Set AORTA_CASE to a label defined in private config/local.py")
cfg = load_case(CASE)

number  = cfg["dataset_id"]
mean_hr = cfg["heart_rate_bpm"]
Psys    = cfg["systolic_mmHg"]  * 1333.22    # mmHg -> dyn/cm^2
Pdia    = cfg["diastolic_mmHg"] * 1333.22    # mmHg -> dyn/cm^2

# Mean outlet flow rates in solver-datatable order. Which branch each slot maps to
# differs between cases; cfg["outlet_branch_order"] records it.
Qavg_out1, Qavg_out2, Qavg_out3, Qavg_out4 = cfg["outlet_mean_flow"]

print(f"case {CASE}: HR {mean_hr} bpm, outlets {cfg['outlet_branch_order']}")


In [ ]:

path_windows = os.path.join(cfg["sim_dir"], f"{number}_ROM", "solver_1d.in")
path_WSL = to_wsl(path_windows)

output_dir_windows = os.path.join(cfg["sim_dir"], f"{number}_ROM") + os.sep
output_dir_WSL = to_wsl(output_dir_windows)

out_name = {
    "out1": f"ROM_{number}_FDbranch1_seg4",
    "out2": f"ROM_{number}_FDbranch2_seg4",
    "out3": f"ROM_{number}_FDbranch3_seg4",
    "out4": f"ROM_{number}_FDbranch4_seg4",
}

In [ ]:
dt_time = 0.001
number_cycles = 8
max_time = 60 / mean_hr  # one cardiac cycle, seconds
steps_per_cycle = time_steps_per_cycle(dt_time, max_time)

Pmean = 2/3*Pdia + 1/3 * Psys
print(
    f"target cycle={max_time:.9g} s, simulated cycle="
    f"{steps_per_cycle * dt_time:.9g} s, steps={steps_per_cycle}, dt={dt_time:.12g} s"
)

# out0
Rp1 = Pmean / Qavg_out1
Rs1 = 0.1 * Rp1
C1 = 1.79 / (Rs1 + Rp1)
# out1
Rp2 = Pmean / Qavg_out2
Rs2 = 0.1 * Rp2
C2 = 1.79 / (Rs2 + Rp2)
# out2
Rp3 = Pmean / Qavg_out3
Rs3 = 0.1 * Rp3
C3 = 1.79 / (Rs3 + Rp3)
# out3
Rp4 = Pmean / Qavg_out4
Rs4 = 0.1 * Rp4
C4 = 1.79 / (Rs4 + Rp4)

outlet_keys = ["out1", "out2", "out3", "out4"]
RCR_params = {
    "out1": {"Rs": Rs1, "Rp": Rp1, "C": C1},
    "out2": {"Rs": Rs2, "Rp": Rp2, "C": C2},
    "out3": {"Rs": Rs3, "Rp": Rp3, "C": C3},
    "out4": {"Rs": Rs4, "Rp": Rp4, "C": C4},
}

print("Best RCR parameters found:")
for key in outlet_keys:
    print(f"  {key} | Rs: {RCR_params[key]['Rs']:.2f}, Rp: {RCR_params[key]['Rp']:.2f}, C: {RCR_params[key]['C']:.6f}")  


In [ ]:
write_in_file_options(path_windows, dt_time, max_time, number_cycles)
clean_previous_runs(output_dir_windows)

In [ ]:

# --- Configuration -----------------------------------------------------------------
max_iterations = 400
convergence_threshold = 0.9   # % — stop when total L2 < this value
stagnation_window     = 10    # stop if L2 doesn't improve by > stagnation_tol over this many accepted iterations
stagnation_tol        = 0.05  # % improvement required to avoid stagnation stop

max_divergence = 6
base_lr = 0.5
min_lr  = 0.0005
lr_shrink_factor = 0.5
lr_grow_factor   = 1.05

total_l2_history = []
total_it_history = []

out_l2_history = {key: [] for key in outlet_keys}

history = {key: {"Rs": [], "Rp": [], "C": [], "L2": []} for key in outlet_keys}

best_RCR_params  = copy.deepcopy(RCR_params)
converged        = False
extra_count      = 10
divergence_count = 0
prev_total_L2    = None
lr_iter          = base_lr


In [ ]:
# Calibration starts in the loop below; no undeclared preliminary run is required.

In [ ]:

# --- Main calibration loop -------------------------------------------------------
counter = 0
for iteration in range(max_iterations):

    output_dir_windows_i = output_dir_windows + f"run_iter{iteration+1}"
    output_dir_WSL_i     = output_dir_WSL     + f"run_iter{iteration+1}"

    if not os.path.exists(output_dir_windows_i):
        os.makedirs(output_dir_windows_i)

    print(f"\n{'='*60}\nIteration {iteration + 1} | LR = {lr_iter:.4f}")

    # 1. Write solver file and run ------------------------------------------------
    write_in_file(
        path_windows,
        [RCR_params["out1"]["Rs"], RCR_params["out1"]["C"], RCR_params["out1"]["Rp"]],
        [RCR_params["out2"]["Rs"], RCR_params["out2"]["C"], RCR_params["out2"]["Rp"]],
        [RCR_params["out3"]["Rs"], RCR_params["out3"]["C"], RCR_params["out3"]["Rp"]],
        [RCR_params["out4"]["Rs"], RCR_params["out4"]["C"], RCR_params["out4"]["Rp"]]
    )
    print("Params sent to solver:")
    for key in outlet_keys:
        print(f"  {key} | Rs:{RCR_params[key]['Rs']:.2f}, Rp:{RCR_params[key]['Rp']:.2f}, C:{RCR_params[key]['C']:.6f}")
    run_result = run_simulation_ubuntu(path_WSL, output_dir=output_dir_WSL_i)

    # 2. Read results -------------------------------------------------------------
    iter_results         = {}
    current_iter_L2_list = []
    for key in outlet_keys:
        p_it, q_it = outlet_pressure_read(out_name[key], output_dir_windows_i, dt_time, max_time, number_cycles)
        Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it = mesure_pressure_characteristics(p_it)
        if key == "out1":
            print(f"  {key} Pressure | Pdist: {Pmin_it/1333.22:.2f}, Psys: {Pmax_it/1333.22:.2f}")
        L, L2_v = L2_mesure(Pmax_it, Pmin_it, Pmean_it, Psys, Pdia, Pmean)
        print(f"  {key} Err | L2:{L2_v*100:.2f}% | Ps:{100*(Pmax_it-Psys)/Psys:5.2f}%, Pd:{100*(Pmin_it-Pdia)/Pdia:5.2f}%, Pm:{100*(Pmean_it-Pmean)/Pmean:5.2f}%")
        iter_results[key] = (p_it, q_it, Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it)
        current_iter_L2_list.append(L2_v)

    total_L2 = (sum(current_iter_L2_list) / len(outlet_keys)) * 100
    print(f"  Total L2 norm: {total_L2:.2f}%")

    # 3. Divergence / acceptance check --------------------------------------------
    skip_update = False

    if prev_total_L2 is not None and total_L2 > prev_total_L2 * 1.05:
        divergence_count += 1
        lr_iter = max(min_lr, lr_iter * lr_shrink_factor)
        print(f"  DIVERGENCE ({divergence_count}/{max_divergence}): L2 {prev_total_L2:.2f}% → {total_L2:.2f}%")
        print(f"  Reverting to best params | New LR = {lr_iter:.5f}")
        RCR_params = copy.deepcopy(best_RCR_params)
        skip_update = True

        if divergence_count >= max_divergence:
            print("  Max divergences reached. Stopping.")
            for key in outlet_keys:
                print(f"  {key} | Rs:{best_RCR_params[key]['Rs']:.2f}, Rp:{best_RCR_params[key]['Rp']:.2f}, C:{best_RCR_params[key]['C']:.6f}")
            print(f"  Total L2 norm: {prev_total_L2:.2f}%")
            break
    else:
        best_RCR_params = copy.deepcopy(RCR_params)
        for key in outlet_keys:
            out_l2_history[key].append(current_iter_L2_list[outlet_keys.index(key)])
        total_l2_history.append(total_L2)
        total_it_history.append(iteration)
        prev_total_L2   = total_L2
        best_iteration  = iteration
        divergence_count = 0
        lr_iter = min(base_lr, lr_iter * lr_grow_factor)
        print(f"  Accepted. | LR adjusted to {lr_iter:.5f}")

    # 4. Newton-Raphson update (skipped after divergence) -------------------------
    if not skip_update:
        for key in outlet_keys:
            p_it, q_it, Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it = iter_results[key]
            Rs_n, Rp_n, C_n = update_rcr_newton_raphson(
                Pmax_it, Pmin_it, Pmean_it, Pmax_idx_it, Pmin_idx_it,
                p_it, q_it, dt_time,
                RCR_params[key]["Rs"], RCR_params[key]["Rp"], RCR_params[key]["C"],
                Psys, Pdia, Pmean,
                learning_rate=lr_iter
            )
            RCR_params[key]["Rs"] = Rs_n
            RCR_params[key]["Rp"] = Rp_n
            RCR_params[key]["C"]  = C_n
        print("  Parameters updated for next iteration.")

    # 5. Convergence check --------------------------------------------------------
    if total_L2 < convergence_threshold:
        converged = True
        print(f"  Convergence achieved! Extra steps: {counter}/{extra_count}")
    if converged:
        counter += 1
        if counter == extra_count:
            print(f"\nStopping: converged for {extra_count} consecutive accepted iterations.")
            break

    # 6. Stagnation check ---------------------------------------------------------
    if len(total_l2_history) >= stagnation_window:
        recent_best = min(total_l2_history[-stagnation_window:])
        older_best  = min(total_l2_history[:-stagnation_window]) if len(total_l2_history) > stagnation_window else total_l2_history[0]
        if older_best - recent_best < stagnation_tol:
            print(f"\nStopping: L2 stagnated (improved < {stagnation_tol}% over last {stagnation_window} iterations).")
            break
      

In [ ]:
print("Best RCR parameters found:")
for key in outlet_keys:
    Rs = best_RCR_params[key]['Rs']
    C = best_RCR_params[key]['C']
    Rp = best_RCR_params[key]['Rp']
    print(f"{key} RCR values: ({Rs:.2f}, {C:.2E}, {Rp:.2f})")

print(f"  with Total L2 norm: {prev_total_L2:.2f}%")



In [ ]:
%matplotlib inline
plt.figure(figsize=(8, 4))
plt.plot(total_it_history, total_l2_history, marker='o')

plt.grid(True)
plt.axhline( 1333.22 *100/Pmean, color='red', linestyle='--', label=f'1 mmHg in Pmean{Pmean}')
plt.legend()
plt.tight_layout()
plt.show()
print(total_it_history)
print(total_l2_history)


In [ ]:
%matplotlib inline
plt.figure(figsize=(8, 4))
for key in outlet_keys:
    plt.plot(total_it_history, [val * 100 for val in out_l2_history[key]], marker='o', label=key)
plt.axhline( 1333.22 *100/Pmean, color='red', linestyle='--', label='1 mmHg')
plt.grid(True)
plt.tight_layout()
plt.show()
print(out_l2_history)

In [ ]:
def plot_last(fname, output_dir_windows, dt_time, max_time,number_cycles,max_iterations):
    number_cycles=number_cycles-1
    fig, axs = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axs = axs.ravel()
    start_dt=int(((max_time/dt_time)*number_cycles)) 
    ax = axs[0]
    
    for outlet in fname.values():
        ax = axs[0]
        pathf = os.path.join(output_dir_windows,f"run_iter{max_iterations}", (outlet+"_pressure.dat"))
        data = np.loadtxt(pathf)
        
        pressure = data[-1, :] / 1333.22  # Convert to mmHg
        time = np.arange(pressure.size) * dt_time
        ax.plot(time, pressure, lw=1)
        

    
        ax.set_xlabel("Time [s]")
        ax.set_ylabel("Pressure")
        ax.grid(True)

        ax = axs[1]
        ax.plot(time[start_dt:] - time[start_dt], pressure[start_dt:], lw=1)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Pressure")
    ax.grid(True)
    
    plt.show()

plot_last(out_name, output_dir_windows, dt_time, max_time,number_cycles,best_iteration)